<a href="https://colab.research.google.com/github/s4140999/cosc2637-task1/blob/main/COSC2637_Part1_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import pandas as pd

click = pd.read_csv('e-shop_clothing_2008.csv', sep=';')
click.columns = [c.strip() for c in click.columns]

print("Shape:", click.shape)
print("Sessions:", click['session ID'].nunique())
print("Missing:", click.isnull().sum().sum())
click.head()

Shape: (165474, 14)
Sessions: 24026
Missing: 0


,year,month,day,order,country,session ID,page 1 (main category),page 2 (clothing model),colour,location,model photography,price,price 2,page
0,2008,4,1,1,29,1,1,A13,1,5,1,28,2,1
1,2008,4,1,2,29,1,1,A16,1,6,1,33,2,1
2,2008,4,1,3,29,1,2,B4,10,2,1,52,1,1
3,2008,4,1,4,29,1,2,B17,6,6,2,38,2,1
4,2008,4,1,5,29,1,2,B8,4,3,2,52,1,1


In [9]:
session = click.groupby('session ID').agg(
    clicks=('order', 'max'),
    n_categories=('page 1 (main category)', 'nunique'),
    avg_price=('price', 'mean'),
    max_page=('page', 'max'),
    n_colours=('colour', 'nunique'),
    month=('month', 'first'),
    pct_above_avg_price=('price 2', lambda x: (x == 1).mean())).reset_index()

median_clicks = session['clicks'].median()
session['deep_engagement'] = (session['clicks'] > median_clicks).astype(int)
print("Sessions:", session.shape)
print("Median clicks:", median_clicks)
print("\nEngagement split:")
print(session['deep_engagement'].value_counts())
session.head()

Sessions: (24026, 9)
Median clicks: 4.0

Engagement split:
deep_engagement
0    13019
1    11007
Name: count, dtype: int64


,session ID,clicks,n_categories,avg_price,max_page,n_colours,month,pct_above_avg_price,deep_engagement
0,1,9,4,42.111111,5,6,4,0.555556,1
1,2,10,3,50.000000,2,5,4,0.800000,1
2,3,6,3,42.166667,5,6,4,0.666667,1
3,4,4,2,45.250000,3,2,4,0.500000,0
4,5,1,1,57.000000,2,1,4,1.000000,0


In [11]:
shoppers = pd.read_csv('online_shoppers_intention.csv')
print("Shape:", shoppers.shape)
print("\nTarget balance:")
print(shoppers['Revenue'].value_counts())
print("\nConversion rate:", round(shoppers['Revenue'].mean()*100,1), "%")
shoppers.head()


Shape: (12330, 18)

Target balance:
Revenue
False    10422
True      1908
Name: count, dtype: int64

Conversion rate: 15.5 %


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False,False
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,True,False


In [12]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

s = pd.get_dummies(shoppers, columns=['Month', 'VisitorType'], drop_first=True)
X = s.drop('Revenue', axis=1)
y = s['Revenue']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

dt = DecisionTreeClassifier(max_depth=5, random_state=42, class_weight='balanced')
dt.fit(X_train, y_train)

rf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf.fit(X_train, y_train)

print("DECISION TREE")
print(classification_report(y_test, dt.predict(X_test), digits=3))
print("ROC-AUC:", round(roc_auc_score(y_test, dt.predict_proba(X_test)[:,1]), 3))
print(confusion_matrix(y_test, dt.predict(X_test)))

print("\nRANDOM FOREST")
print(classification_report(y_test, rf.predict(X_test), digits=3))
print("ROC-AUC:", round(roc_auc_score(y_test, rf.predict_proba(X_test)[:,1]), 3))
print(confusion_matrix(y_test, rf.predict(X_test)))

DECISION TREE
              precision    recall  f1-score   support

       False      0.965     0.831     0.893      3127
        True      0.474     0.834     0.604       572

    accuracy                          0.831      3699
   macro avg      0.719     0.832     0.748      3699
weighted avg      0.889     0.831     0.848      3699

ROC-AUC: 0.904
[[2597  530]
 [  95  477]]

RANDOM FOREST
              precision    recall  f1-score   support

       False      0.918     0.968     0.942      3127
        True      0.750     0.530     0.621       572

    accuracy                          0.900      3699
   macro avg      0.834     0.749     0.782      3699
weighted avg      0.892     0.900     0.893      3699

ROC-AUC: 0.917
[[3026  101]
 [ 269  303]]


In [13]:
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("Top 10 features (Random Forest):")
print(importances.head(10))

Top 10 features (Random Forest):
PageValues                 0.383305
ExitRates                  0.099711
ProductRelated_Duration    0.093113
ProductRelated             0.066216
BounceRates                0.058309
Administrative_Duration    0.051504
Administrative             0.034882
Month_Nov                  0.027426
TrafficType                0.026947
Region                     0.025835
dtype: float64


In [15]:
X2 = session.drop(['session ID', 'clicks', 'deep_engagement'], axis=1)
y2 = session['deep_engagement']

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.3, random_state=42, stratify=y2)

dt2 = DecisionTreeClassifier(max_depth=5, random_state=42)
dt2.fit(X2_train, y2_train)

rf2 = RandomForestClassifier(n_estimators=100, random_state=42)
rf2.fit(X2_train, y2_train)

print("DECISION TREE (clickstream)")
print(classification_report(y2_test, dt2.predict(X2_test), digits=3))
print("ROC-AUC:", round(roc_auc_score(y2_test, dt2.predict_proba(X2_test)[:,1]), 3))

print("\nRANDOM FOREST (clickstream)")
print(classification_report(y2_test, rf2.predict(X2_test), digits=3))
print("ROC-AUC:", round(roc_auc_score(y2_test, rf2.predict_proba(X2_test)[:,1]), 3))

print("\nTop features:")
print(pd.Series(rf2.feature_importances_, index=X2.columns).sort_values(ascending=False))

DECISION TREE (clickstream)
              precision    recall  f1-score   support

           0      0.911     0.959     0.934      3906
           1      0.948     0.890     0.918      3302

    accuracy                          0.927      7208
   macro avg      0.930     0.924     0.926      7208
weighted avg      0.928     0.927     0.927      7208

ROC-AUC: 0.973

RANDOM FOREST (clickstream)
              precision    recall  f1-score   support

           0      0.939     0.964     0.951      3906
           1      0.956     0.926     0.941      3302

    accuracy                          0.946      7208
   macro avg      0.947     0.945     0.946      7208
weighted avg      0.947     0.946     0.946      7208

ROC-AUC: 0.985

Top features:
n_colours              0.454453
pct_above_avg_price    0.178044
n_categories           0.158227
avg_price              0.118547
max_page               0.069094
month                  0.021635
dtype: float64


In [16]:
# Confusion matrices for the clickstream models
print("DT:\n", confusion_matrix(y2_test, dt2.predict(X2_test)))
print("\nRF:\n", confusion_matrix(y2_test, rf2.predict(X2_test)))

DT:
 [[3744  162]
 [ 364 2938]]

RF:
 [[3765  141]
 [ 245 3057]]
